# Assignment 4 – Data Processing

**Name:** Putri Redita Rositasari  
**Student ID:** 11570018

## Research Question

How does relative humidity (RH) vary with air temperature (AMB_TEMP) in Taiwan?

## Data Source

Taiwan Ministry of Environment (MOENV) Open Data API  
Dataset: AQX_P_35

## Selected Variables

- AMB_TEMP: Air Temperature (°C)
- RH: Relative Humidity (%)

In [4]:
import pandas as pd
import numpy as np
from pathlib import Path

print("Pandas version:", pd.__version__)

Pandas version: 2.2.3


In [5]:
import getpass

API_KEY = getpass.getpass("Enter your MOENV API key: ")
print("API key loaded.")

Enter your MOENV API key: ··········
API key loaded.


In [6]:
import requests
import pandas as pd
import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

url = "https://data.moenv.gov.tw/api/v2/aqx_p_35"

params = {
    "limit": 1000,
    "format": "json",
    "api_key": API_KEY,
}

response = requests.get(
    url,
    params=params,
    timeout=30,
    verify=False
)

response.raise_for_status()

payload = response.json()

print("HTTP status:", response.status_code)
print("Payload type:", type(payload))

HTTP status: 200
Payload type: <class 'list'>


In [7]:
df = pd.DataFrame(payload)

In [8]:
print("Dataset shape:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

df.head()

Dataset shape: (1000, 9)

Column names:
['siteid', 'sitename', 'itemid', 'itemname', 'itemengname', 'itemunit', 'monitordate', 'concentration', 'county']


,siteid,sitename,itemid,itemname,itemengname,itemunit,monitordate,concentration,county
0,40,朴子,14,溫度,AMB_TEMP,℃,2026-10-14 20:00,26.7,嘉義縣
1,53,楠梓,14,溫度,AMB_TEMP,℃,2026-10-14 20:00,28.4,高雄市
2,41,臺西,38,相對濕度,RH,percent,2026-10-14 20:00,68,雲林縣
3,45,安南,38,相對濕度,RH,percent,2026-10-14 20:00,66,臺南市
4,39,新港,38,相對濕度,RH,percent,2026-10-14 20:00,66,嘉義縣


## 2. Select Relevant Variables

Only air temperature (AMB_TEMP) and relative humidity (RH) are required to answer the research question. The site and monitoring time variables are retained so that temperature and humidity observations can be matched correctly.

In [9]:
# Keep only AMB_TEMP and RH observations
selected_df = df[
    df["itemengname"].isin(["AMB_TEMP", "RH"])
].copy()

# Keep only relevant columns
selected_df = selected_df[
    ["siteid", "sitename", "county", "monitordate",
     "itemengname", "itemunit", "concentration"]
]

print("Original rows:", len(df))
print("Rows after selecting AMB_TEMP and RH:", len(selected_df))

selected_df.head(10)

Original rows: 1000
Rows after selecting AMB_TEMP and RH: 1000


,siteid,sitename,county,monitordate,itemengname,itemunit,concentration
0,40,朴子,嘉義縣,2026-10-14 20:00,AMB_TEMP,℃,26.7
1,53,楠梓,高雄市,2026-10-14 20:00,AMB_TEMP,℃,28.4
2,41,臺西,雲林縣,2026-10-14 20:00,RH,percent,68
3,45,安南,臺南市,2026-10-14 20:00,RH,percent,66
4,39,新港,嘉義縣,2026-10-14 20:00,RH,percent,66
5,50,鳳山,高雄市,2026-10-14 20:00,AMB_TEMP,℃,29
6,29,沙鹿,臺中市,2026-10-14 20:00,RH,percent,58
7,50,鳳山,高雄市,2026-10-14 20:00,RH,percent,68
8,33,彰化,彰化縣,2026-10-14 20:00,AMB_TEMP,℃,27
9,23,竹東,新竹縣,2026-10-14 20:00,RH,percent,56


## 3. Data Cleaning

The concentration values are converted to numeric format. Invalid or missing values are identified and removed to prepare the data for analysis.

In [10]:
# Convert concentration to numeric
selected_df["concentration"] = pd.to_numeric(
    selected_df["concentration"],
    errors="coerce"
)

# Check missing values after numeric conversion
print("Missing values before removal:")
print(selected_df.isna().sum())

print("\nRows before removing missing values:", len(selected_df))

# Remove rows with missing concentration
clean_df = selected_df.dropna(subset=["concentration"]).copy()

print("Rows after removing missing values:", len(clean_df))

clean_df.head(10)

Missing values before removal:
siteid            0
sitename          0
county            0
monitordate       0
itemengname       0
itemunit          0
concentration    12
dtype: int64

Rows before removing missing values: 1000
Rows after removing missing values: 988


,siteid,sitename,county,monitordate,itemengname,itemunit,concentration
0,40,朴子,嘉義縣,2026-10-14 20:00,AMB_TEMP,℃,26.7
1,53,楠梓,高雄市,2026-10-14 20:00,AMB_TEMP,℃,28.4
2,41,臺西,雲林縣,2026-10-14 20:00,RH,percent,68.0
3,45,安南,臺南市,2026-10-14 20:00,RH,percent,66.0
4,39,新港,嘉義縣,2026-10-14 20:00,RH,percent,66.0
5,50,鳳山,高雄市,2026-10-14 20:00,AMB_TEMP,℃,29.0
6,29,沙鹿,臺中市,2026-10-14 20:00,RH,percent,58.0
7,50,鳳山,高雄市,2026-10-14 20:00,RH,percent,68.0
8,33,彰化,彰化縣,2026-10-14 20:00,AMB_TEMP,℃,27.0
9,23,竹東,新竹縣,2026-10-14 20:00,RH,percent,56.0


In [11]:
pd.to_numeric(..., errors="coerce")

Ellipsis

## 4. Create an Analysis-Ready Table

The cleaned data are reshaped from long format to wide format so that AMB_TEMP and RH measurements from the same monitoring site and time appear in the same row.

In [12]:
# Reshape from long format to wide format
analysis_ready = clean_df.pivot_table(
    index=["siteid", "sitename", "county", "monitordate"],
    columns="itemengname",
    values="concentration",
    aggfunc="first"
).reset_index()

# Remove the column index name
analysis_ready.columns.name = None

print("Shape before reshaping:", clean_df.shape)
print("Shape after reshaping:", analysis_ready.shape)

analysis_ready.head(10)

Shape before reshaping: (988, 7)
Shape after reshaping: (512, 6)


,siteid,sitename,county,monitordate,AMB_TEMP,RH
0,1,基隆,基隆市,2026-10-14 14:00,27.7,NaN
1,1,基隆,基隆市,2026-10-14 15:00,27.4,65.0
2,1,基隆,基隆市,2026-10-14 16:00,26.5,71.0
3,1,基隆,基隆市,2026-10-14 17:00,26.0,73.0
4,1,基隆,基隆市,2026-10-14 18:00,25.3,79.0
5,1,基隆,基隆市,2026-10-14 19:00,24.8,82.0
6,1,基隆,基隆市,2026-10-14 20:00,25.5,76.0
7,10,淡水,新北市,2026-10-14 14:00,27.8,55.0
8,10,淡水,新北市,2026-10-14 15:00,26.6,59.0
9,10,淡水,新北市,2026-10-14 16:00,25.5,62.0


## 5. Check Data Completeness

Rows with missing AMB_TEMP or RH values are identified. Only complete temperature–humidity pairs are retained for the final analysis-ready dataset.

In [13]:
# Check missing values in the reshaped table
print("Missing values:")
print(analysis_ready[["AMB_TEMP", "RH"]].isna().sum())

print("\nRows before removing incomplete pairs:", len(analysis_ready))

# Keep only complete AMB_TEMP-RH pairs
final_df = analysis_ready.dropna(
    subset=["AMB_TEMP", "RH"]
).copy()

print("Rows after removing incomplete pairs:", len(final_df))

final_df.head(10)

Missing values:
AMB_TEMP     1
RH          35
dtype: int64

Rows before removing incomplete pairs: 512
Rows after removing incomplete pairs: 476


,siteid,sitename,county,monitordate,AMB_TEMP,RH
1,1,基隆,基隆市,2026-10-14 15:00,27.4,65.0
2,1,基隆,基隆市,2026-10-14 16:00,26.5,71.0
3,1,基隆,基隆市,2026-10-14 17:00,26.0,73.0
4,1,基隆,基隆市,2026-10-14 18:00,25.3,79.0
5,1,基隆,基隆市,2026-10-14 19:00,24.8,82.0
6,1,基隆,基隆市,2026-10-14 20:00,25.5,76.0
7,10,淡水,新北市,2026-10-14 14:00,27.8,55.0
8,10,淡水,新北市,2026-10-14 15:00,26.6,59.0
9,10,淡水,新北市,2026-10-14 16:00,25.5,62.0
10,10,淡水,新北市,2026-10-14 17:00,24.5,72.0


## 6. Save the Analysis-Ready Table

The final processed dataset contains complete paired observations of air temperature and relative humidity for each monitoring site and time.

In [14]:
# Save the final analysis-ready table
output_file = "analysis_ready_temperature_humidity.csv"

final_df.to_csv(output_file, index=False)

print("Processed output table:", output_file)
print("Final number of rows:", len(final_df))
print("Final number of columns:", len(final_df.columns))

final_df.head()

Processed output table: analysis_ready_temperature_humidity.csv
Final number of rows: 476
Final number of columns: 6


,siteid,sitename,county,monitordate,AMB_TEMP,RH
1,1,基隆,基隆市,2026-10-14 15:00,27.4,65.0
2,1,基隆,基隆市,2026-10-14 16:00,26.5,71.0
3,1,基隆,基隆市,2026-10-14 17:00,26.0,73.0
4,1,基隆,基隆市,2026-10-14 18:00,25.3,79.0
5,1,基隆,基隆市,2026-10-14 19:00,24.8,82.0


## 7. Processing Audit Table

The following table summarizes the major changes made to the dataset during processing.

In [15]:
audit_table = pd.DataFrame({
    "Processing Step": [
        "Raw API data",
        "Select AMB_TEMP and RH",
        "Remove invalid/missing concentration",
        "Reshape to paired observations",
        "Remove incomplete AMB_TEMP-RH pairs"
    ],
    "Rows After Step": [
        len(df),
        len(selected_df),
        len(clean_df),
        len(analysis_ready),
        len(final_df)
    ]
})

audit_table["Rows Removed"] = [
    0,
    len(df) - len(selected_df),
    len(selected_df) - len(clean_df),
    "N/A",
    len(analysis_ready) - len(final_df)
]

audit_table

,Processing Step,Rows After Step,Rows Removed
0,Raw API data,1000,0
1,Select AMB_TEMP and RH,1000,0
2,Remove invalid/missing concentration,988,12
3,Reshape to paired observations,512,N/A
4,Remove incomplete AMB_TEMP-RH pairs,476,36


In [18]:
final_df.to_csv(
    "analysis_ready_temperature_humidity.csv",
    index=False,
    encoding="utf-8-sig"
)

print("File saved with UTF-8 encoding!")

File saved with UTF-8 encoding!


## 8. Short Interpretation

### What processing step changed your data the most?

The reshaping step changed the structure of the data the most. The dataset was transformed from long format into a wide format, placing AMB_TEMP and RH measurements from the same monitoring site and time in a single row. This reduced the table from 988 measurement rows to 512 paired-observation rows.

### What processing assumption could affect your result?

The processing assumes that AMB_TEMP and RH measurements with the same monitoring site and monitoring time represent corresponding observations and can therefore be paired for analysis.

### What check could detect a processing error?

Checking missing values, row counts, duplicate site-time combinations, and the expected ranges of AMB_TEMP and RH can help detect errors introduced during processing.